# 2026-07-22 — five cities on v2, PC predictors enter Stage 2

## goal
K=6 on v2, Stage 2 with PCs, PC slopes non-zero.

## methods
`run_city` built: csv → substrate_v2 → per-DA MMT → da_long → sim → 150-DA sliver → cross-basis → Stage 1 → reduce. Returns red/res/diag/Z/da_ids; sim dropped at the sliver cut, substrate after the reduce.

Five cities through it at seed 42 + offset: Toronto 35, Montréal 24, Vancouver 59, Ottawa 63, Québec 421. Ottawa and Québec populations built first — Ontario and Quebec profiles, DGRF filter, streaming grep, four age bands. Ottawa greps both provinces and stacks.

CMA-level predictors from column means of each city's Z, 5 × 17, `prcomp(scale. = TRUE)`, PC1–3 into `fit_stage2`.

## results
fns.R 406 lines, 17 functions, landmines 0. +68 lines unlogged since 07-15.

OTT/CAL/QC daymet row counts 1,564,426 / 1,451,971 / 1,007,506 — DA × 765 + 1 header.

Ottawa 2,045 DAs / 1,487,965, bands 83.0 / 9.9 / 5.0 / 2.0. Québec City 1,317 / 839,300, bands 78.4 / 12.3 / 6.8 / 2.5.

Toronto through `run_city` reproduces the 07-15 hand-assembled run: μ [-0.330, 0.655, 0.384], 610,216 deaths, qAIC A 27,047.0 / B 55,301.9, θ* [-0.843, -6.486, -5.099, -6.053, -3.616], ref_temp 19.38.

| CMA | n_da | μ | deaths | sliver | ref_temp | qAIC A |
|---|---|---|---|---|---|---|
| Toronto | 7682 | [-0.330, 0.655, 0.384] | 610,216 | 2,419 | 19.38 | 27,047.0 |
| Montréal | 6504 | [1.394, 0.130, 0.251] | 203,988 | 1,524 | 19.01 | 27,967.1 |
| Vancouver | 3573 | [-0.196, 0.331, -0.405] | 56,157 | 649 | 17.05 | 27,441.6 |
| Ottawa | 2042 | [-0.776, -0.280, 0.080] | 51,476 | 1,247 | 18.58 | 28,992.7 |
| Québec | 1310 | [-0.077, -1.059, 0.200] | 26,977 | 982 | 16.79 | 28,083.3 |

All Variant A, all V_star positive-definite, μ duplicate check 0.

Québec first ran at offset 24 and returned μ [1.394, 0.130, 0.251] — Montréal's, exact. Re-run at 421: deaths 39,207 → 26,977, n_da and ref_temp unchanged to the digit.

Z_means 5 × 17, rows distinct. Per-PC 0.431 / 0.336 / 0.233, PC4 zero.

Stage 2 K=3, intercept-only: REML converged, Cholesky FALSE, df.residual −5, pooled [0.442, -4.614, -3.403, -5.038, 1.07].

Stage 2 K=5, ~ PC1 + PC2 + PC3: REML converged, Cholesky TRUE, 20 coefficients, df.residual −10. PC1 spans -1.448 to +0.165, PC3 +0.462 to +1.118.

Banked to saves_eod_2026-07-22: red5_v2, res5_v2, Z5_v2, ids5_v2, diag5_v2, stage2_k5_v2, cma_predictors, new_age_ottqc. SSOT pushed, seven edits.

## discussion
Québec City is older than Ottawa — 78.4% under 65 against 83.0, 6.8% in the 75–84 band against 5.0. Proportionally more population at risk in the band we model.

Ottawa's 2,042 DAs carry 1,247 sliver deaths where Vancouver's 3,573 carry 649. The sliver is 150 DAs regardless of city size, so this is per-DA 75–84 population.

Deaths span 610k to 27k across a 2.2× DA range — population, not DA count. V_star diagonals scale inversely with sliver deaths, so Vancouver contributes least to the pool at 649.

The multiplier against v1 runs through μ₂ and `exp(0.3·F2·cold)`: Toronto +0.655 → 3.2×, MTL +0.130 → 1.43×, VAN +0.331 → 1.02×, Québec −1.059 shrinks. Not monotone in μ₂ — Vancouver is maritime, fewer days below its own MMT.

A wins every city because B splits into 26,250 strata against A's 3,750; MTL runs 0.06 deaths per stratum under B against 0.41 under A. Sliver artifact.

`fit_stage1` raised `object 'cb' not found` on both variants inside `run_city`. The rule was that the cross-basis stays out of the model frame and gnm resolves `cb` from the calling environment. The second half only held at top level — inside a wrapper `cb` is a local and the formula's environment chain never reaches it. A rule verified at top level is not verified inside a function.

The seed convention `42 + province prefix` was correct for every city that existed and wrong by construction. Montréal and Québec City are both province 24. A key unique at the current N is not a key; CMA code is unique by design.

Alberta has no DA-level Census Profile. The DA files are per-region—Atlantic, Quebec, Ontario, Prairies, BC, Territories—and Alberta sits inside Prairies. `006_Ontario` and `006_Quebec` work because those provinces are their own regions, which reads like a naming convention and isn't.

Ψ failed Cholesky at K=3 and passed at K=5. At K=3 all between-city spread had to live in Ψ, 15 free parameters from 3 groups; the PCs absorb part of it into fixed effects. `df.residual` went −5 → −10 when three predictors entered, so more cities alone will not clear it.

## handoff
Calgary: geography and temperature exist, population does not. Read the Prairies GEONO off the Census Profile download page, run §9.7b logic with a third province, `seed_offset = 825`.

Stage 3 downscale. `predict_da_theta` (§8.2) needs the PC-predictor form — per-DA θ from the DA's own PC score. `compute_da_mmt` two-pass per DA. Recovery target `exp(0.4·F1 + 0.2·F3)`, scored on amplitude, success is recovered SD > 0. §8.8's panel is v1 additive, N_CMA=3, intercept-only.

§7.5 consumes all five `*_out` objects and must run after §9.8 — check its cell placement.

fns.R grew 68 lines with no session record. Audit passes.

λ max 9,759.7 against λ₀ max 0.1788, carried from 07-15. Not a blocker.

§0 restore. library cache, re-attach, load() then source(fns.R).

In [ ]:
DRIVE <- "/content/drive/MyDrive/thesis/dlnm-pilot"

system(sprintf("cd /content && cp %s/r_library.tar.gz . && tar -xzf r_library.tar.gz", DRIVE))
.libPaths(c("/content/site-library", .libPaths()))

suppressPackageStartupMessages({
  library(dlnm); library(gnm); library(mixmeta); library(splines)
  library(sf); library(data.table); library(exactextractr); library(terra)
  library(ggplot2); library(viridis); library(lubridate)
})

system(sprintf("cd /content && cp %s/saves_pilot_2026-06-03.tar.gz . && tar -xzf saves_pilot_2026-06-03.tar.gz", DRIVE))
load("/content/saves/pilot_session.RData")

source(file.path(DRIVE, "fns.R"))

year_start <- 2015; year_end <- 2019; warm_months <- 5:9
study_dates <- seq.Date(as.Date(sprintf("%d-05-01", year_start)),
                        as.Date(sprintf("%d-09-30", year_end)), by = "day")
study_dates <- study_dates[lubridate::month(study_dates) %in% warm_months]

need <- c("read_daymet","build_crossbasis","make_strata_A","make_strata_B","qaic",
          "fit_city_sliver","build_city_sim_substrate_v2","simulate_counts",
          "fit_stage1","reduce_fit","fit_stage2","fit_da_pca","predict_da_theta",
          "compute_da_mmt","monte_carlo_ci","standardize_da_rate","save_to_drive")
fns_txt <- readLines(file.path(DRIVE, "fns.R"))
audit <- data.table(fn = need,
  in_env  = sapply(need, function(f) exists(f) && is.function(get(f))),
  in_file = sapply(need, function(f) any(grepl(sprintf("^%s <- function", f), fns_txt))))
audit[, landmine := in_env & !in_file]

cat("landmines:", sum(audit$landmine), " (stop if nonzero)\n")
cat("missing from file:", paste(audit[in_file == FALSE]$fn, collapse=", "), "\n")
cat("simulate_counts has exp:",
    grepl("exp(0.4", paste(deparse(body(simulate_counts)), collapse=" "), fixed = TRUE), "\n")
cat("study_dates:", length(study_dates), " L dim:", paste(dim(L), collapse=" x "), "\n")
cat("DGP primitives:", all(sapply(c("base_log_rr","lag_weights","L","da_age","annual_rates","daymet_csv"), exists)), "\n")
cat("fns.R lines:", length(fns_txt), " (was 338)\n")

landmines: 0  (stop if nonzero)
missing from file:  
simulate_counts has exp: TRUE 
study_dates: 765  L dim: 17 x 3 
DGP primitives: TRUE 
fns.R lines: 406  (was 338)


landmines: 0  (stop if nonzero)
missing from file:  
simulate_counts has exp: TRUE
study_dates: 765  L dim: 17 x 3
DGP primitives: TRUE
fns.R lines: 406  (was 338)

---
nothing missing from file, the DGP fix survived. study_dates
765 = 153 warm days x 5 yrs, L 17x3,

fns.R 406 lines, not the 338, so +68 unlogged. check later

state: packages live, 17 fns sourced, DGP primitives in env, study_dates built.


ott/cal/qc daymet csvs — exist or not; 07-08 log says exported, ssot §9.8 says drafted. predict ott 1564425 (2045x765), cal 1451970 (1898x765), qc 1007505 (1317x765)+1 header) = DA x 765.

file.exists loop reports absent on files that are sitting there if export named differently

In [ ]:
system(sprintf("ls -la %s/*daymet*.csv", DRIVE))

for (city in c("ottawa","calgary","quebec")) {
  f <- sprintf("%s/%s_daymet_2015_2019.csv", DRIVE, city)
  if (file.exists(f)) {
    n <- as.integer(system(sprintf("wc -l < '%s'", f), intern = TRUE))
    cat(sprintf("%-9s rows: %8d\n", city, n))
  } else {
    cat(sprintf("%-9s ABSENT\n", city))
  }
}

ottawa    rows:  1564426
calgary   rows:  1451971
quebec    rows:  1007506


ottawa    rows:  1564426
calgary   rows:  1451971
quebec    rows:  1007506

---

all three csvs real. ott 1564426 / cal 1451971 / qc 1007506 — each = DA x 765 + 1 header,
exact.

07-08 log right, ssot §9.8 stale (says "drafted"). K=6 reachable.

run_city — csv to reduced curve, one call. read_daymet → substrate_v2 → mmt → da_long →
sim → 150-DA sliver → crossbasis → fit_stage1 → reduce_fit

returns red + res + diag row only. sim stays local and dies on exit — last time the return
held substrate+sim+sliver and two 20M-row tables took the kernel at 12.7gb

defining only no numbers yet

In [ ]:
run_city <- function(csv_name, da_age_city, cma_label, seed_offset,
                     L, age = "age_75_84", n_sliver = 150, verbose = TRUE) {

  if (verbose) cat(sprintf("\n===== %s =====\n", cma_label))

  dm  <- read_daymet(file.path(DRIVE, csv_name))
  sub <- build_city_sim_substrate_v2(dm, da_age_city, L, seed = 42 + seed_offset)
  rm(dm); gc(verbose = FALSE)

  stopifnot(all(rownames(sub$temp_mat) == sub$truth_factors$DAUID))
  if (verbose) cat(sprintf("  n_da %d  mu [%s]  temp NA %d\n",
                           sub$n_da, paste(round(sub$mu, 3), collapse=", "),
                           sum(is.na(sub$temp_mat))))

  mmt <- apply(sub$temp_mat, 1, function(x) quantile(x, 0.80, na.rm = TRUE))

  dal <- CJ(da_idx = 1:sub$n_da, date = study_dates, age_band = names(annual_rates))
  pl  <- melt(sub$da_age[, .(da_idx, age_0_64, age_65_74, age_75_84, age_85p)],
              id.vars = "da_idx", variable.name = "age_band", value.name = "pop")
  pl[, age_band := as.character(age_band)]
  dal <- pl[dal, on = c("da_idx","age_band")]
  dal[, annual_rate := annual_rates[age_band]]
  dal[, lambda0 := pop * annual_rate / 1000 / 365]
  stopifnot(sum(is.na(dal$pop)) == 0)

  sim <- simulate_counts(sub$temp_mat, dal, sub$truth_factors, mmt, seed = 42)
  rm(dal, pl); gc(verbose = FALSE)
  tot_deaths <- sum(sim$n_deaths)
  if (verbose) cat(sprintf("  deaths %s  lambda NA/Inf %d/%d  pct zero %.2f\n",
                           format(tot_deaths, big.mark=","),
                           sum(is.na(sim$lambda)), sum(is.infinite(sim$lambda)),
                           100*mean(sim$n_deaths == 0)))

  set.seed(42)
  idx <- sample(unique(sim$da_idx), n_sliver)
  sl  <- sim[da_idx %in% idx & age_band == age]
  sl[, DA_id := da_idx]
  setorder(sl, da_idx, date)
  rm(sim); gc(verbose = FALSE)

  tl <- data.table(da_idx = rep(idx, each = length(study_dates)),
                   date   = rep(study_dates, times = n_sliver),
                   temp_C = as.vector(t(sub$temp_mat[idx, ])))
  sl <- tl[sl, on = c("da_idx","date")]
  stopifnot(nrow(sl) == n_sliver * length(study_dates), sum(is.na(sl$temp_C)) == 0)

  cb <- build_crossbasis(sl$temp_C, lag_max = 21)
  dt <- data.table(n_deaths = sl$n_deaths, DA_id = sl$DA_id, date = sl$date)
  stopifnot(nrow(dt) == nrow(cb))

  res <- fit_stage1(dt, cb, cma_label, age, verbose = verbose)
  if (!identical(res$status, "ok")) {
    if (verbose) cat("  stage 1 failed\n")
    return(list(cma = cma_label, status = "failed", res = res))
  }

  ref  <- median(sl$temp_C)
  red  <- reduce_fit(res, ref)

  diag <- data.table(cma = cma_label, n_da = sub$n_da,
                     mu1 = sub$mu[1], mu2 = sub$mu[2], mu3 = sub$mu[3],
                     deaths = tot_deaths, sliver_deaths = sum(sl$n_deaths),
                     ref_temp = ref, winner = res$winner_variant,
                     qaic_A = res$qaic_A, qaic_B = res$qaic_B)

  Zc <- sub$Z; da_ids <- sub$truth_factors$DAUID
  rm(sub, sl, tl, dt); gc(verbose = FALSE)

  list(cma = cma_label, status = "ok", red = red, res = res,
       diag = diag, Z = Zc, da_ids = da_ids)
}

body_txt <- paste(deparse(body(run_city)), collapse = " ")
cat("run_city defined:", exists("run_city"), "\n")
cat("rm(sim) present:", grepl("rm(sim)", body_txt, fixed = TRUE), "\n")
cat("returns substrate:", grepl("sub = sub", body_txt, fixed = TRUE), " (must be FALSE)\n")
cat("returns sim:", grepl("sim = sim", body_txt, fixed = TRUE), " (must be FALSE)\n")
cat("calls _v2 builder:", grepl("build_city_sim_substrate_v2", body_txt, fixed = TRUE), "\n")

run_city defined: TRUE 
rm(sim) present: TRUE 
returns substrate: FALSE  (must be FALSE)
returns sim: FALSE  (must be FALSE)
calls _v2 builder: TRUE 


run_city defined: TRUE
rm(sim) present: TRUE
returns substrate: FALSE  (must be FALSE)
returns sim: FALSE  (must be FALSE)
calls _v2 builder: TRUE  

---

run_city defined. rm(sim) present, return carries neither substrate nor sim, calls the _v2
builder not v1

mtl through run_city. chain: csv → dm → sub → mmt → dal → sim → sl(150 DA) → cb → res → red
cma_age_data_mtlvan pulled from saves_eod — population, builder-agnostic so safe to reuse

predict: mu three distinct nonzero and different from tor's [-0.33, 0.655, 0.384] (read first;
same mu = seed not varying), n_da 6504, deaths 300-700k vs v1's 142,934, winner A

In [ ]:
system(sprintf("cd /content && cp %s/saves_eod_2026-06-25.tar.gz . && tar -xzf saves_eod_2026-06-25.tar.gz", DRIVE))
cma_age_data_mtlvan <- readRDS("/content/saves_eod/cma_age_data_mtlvan.rds")

cat("names:", paste(names(cma_age_data_mtlvan), collapse=", "), "\n")
cat("mtl rows:", nrow(cma_age_data_mtlvan$montreal),
    " van rows:", nrow(cma_age_data_mtlvan$vancouver), "\n")

mtl_out <- run_city("montreal_daymet_2015_2019.csv",
                    cma_age_data_mtlvan$montreal,
                    "Montreal", seed_offset = 24, L = L)

cat("\nstatus:", mtl_out$status, "\n")
print(mtl_out$diag)
cat("theta:", paste(round(mtl_out$red$theta_star, 3), collapse=" "), "\n")
cat("V diag:", paste(round(diag(mtl_out$red$V_star), 3), collapse=" "), "\n")
cat("V pos-def:", tryCatch({ chol(mtl_out$red$V_star); TRUE }, error = function(e) FALSE), "\n")
cat("Z:", paste(dim(mtl_out$Z), collapse=" x "), " da_ids:", length(mtl_out$da_ids), "\n")
cat("mem gb:", round(sum(gc()[,2])/1024, 2), "\n")

names: montreal, vancouver 
mtl rows: 6574  van rows: 3590 

===== Montreal =====
  n_da 6504  mu [1.394, 0.13, 0.251]  temp NA 0
  deaths 203,988  lambda NA/Inf 0/0  pct zero 99.17

=== Stage 1: Montreal, age age_75_84 ===
  Variant A: gnm RAISED -> object 'cb' not found
  Variant B: gnm RAISED -> object 'cb' not found
  BOTH VARIANTS FAILED — returning diagnostics, no fit
  stage 1 failed

status: failed 
NULL


ERROR: Error in round(mtl_out$red$theta_star, 3): non-numeric argument to mathematical function


names: montreal, vancouver
mtl rows: 6574  van rows: 3590

===== Montreal =====
  n_da 6504  mu [1.394, 0.13, 0.251]  temp NA 0
  deaths 203,988  lambda NA/Inf 0/0  pct zero 99.17

=== Stage 1: Montreal, age age_75_84 ===
  Variant A: gnm RAISED -> object 'cb' not found
  Variant B: gnm RAISED -> object 'cb' not found
  BOTH VARIANTS FAILED — returning diagnostics, no fit
  stage 1 failed

status: failed
NULL
Error in round(mtl_out$red$theta_star, 3): non-numeric argument to mathematical function
Traceback:

1. paste(round(mtl_out$red$theta_star, 3), collapse = " ")
2. .handleSimpleError(function (cnd)
 . {
 .     watcher$capture_plot_and_output()
 .     cnd <- sanitize_call(cnd)
 .     watcher$push(cnd)
 .     switch(on_error, continue = invokeRestart("eval_continue"),
 .         stop = invokeRestart("eval_stop"), error = NULL)
 . }, "non-numeric argument to mathematical function", base::quote(round(mtl_out$red$theta_star,
 .     3)))

---

mtl substrate + sim clean. n_da 6504, mu [1.394, 0.13, 0.251], distinct from tor's = seed
varies per city, between-CMA variance. deaths 203,988 vs v1 142,934 = 1.43x not tor's
3.2x. mu2 drives it: tor 0.655, mtl 0.13, inflation runs through exp(0.3·F2·cold).

stage 1 raised both variants, object 'cb' not found. "cb out
of the frame" holds, "gnm resolves from calling env" only held because everything ran at top
level. inside run_city cb is a local and the formula's env chain doesn't reach a wrapper frame.

predicted winner A, got a raise. wrapping top-level code in a function moved the
environment out from under gnm.

bind cb_template to cb inside the frame the formula is built in
one line, cb <- cb_template at the top

In [ ]:
fit_stage1 <- function(cma_age_data, cb_template, cma_label, age_label, verbose = TRUE) {
  if (verbose) cat(sprintf("\n=== Stage 1: %s, age %s ===\n", cma_label, age_label))

  cb <- cb_template

  cma_age_data[, strata_A := make_strata_A(DA_id, date)]
  cma_age_data[, strata_B := make_strata_B(DA_id, date)]
  cma_age_data[, dow := factor(wday(date))]
  cma_age_data[, t   := as.integer(date - min(date)) + 1]

  results <- list()
  for (variant in c("A", "B")) {
    strata_col <- if (variant == "A") "strata_A" else "strata_B"
    formula <- if (variant == "A") {
      n_deaths ~ cb + dow + ns(t, df = 20)
    } else {
      n_deaths ~ cb + ns(t, df = 20)
    }
    cma_age_data[, strata_use := get(strata_col)]

    fit <- try(gnm(formula, data = cma_age_data,
                   family = quasipoisson(), eliminate = strata_use),
               silent = TRUE)

    if (inherits(fit, "try-error")) {
      err <- conditionMessage(attr(fit, "condition"))
      if (verbose) cat(sprintf("  Variant %s: gnm raised -> %s\n", variant, err))
      results[[variant]] <- list(status = "raised", err = err)
      next
    }
    if (!isTRUE(fit$converged)) {
      if (verbose) cat(sprintf("  Variant %s: gnm ran, converged FALSE (iterMax)\n", variant))
      results[[variant]] <- list(status = "noconv", fit = fit)
      next
    }

    results[[variant]] <- list(
      status   = "ok",
      fit      = fit,
      qaic     = qaic(fit),
      n_strata = length(unique(cma_age_data[[strata_col]])),
      mean_deaths_per_stratum = sum(cma_age_data$n_deaths) /
                                length(unique(cma_age_data[[strata_col]]))
    )
    if (verbose) cat(sprintf("  Variant %s: converged, qAIC = %.1f, strata = %d\n",
                             variant, results[[variant]]$qaic, results[[variant]]$n_strata))
  }

  ok <- names(results)[sapply(results, function(r) r$status == "ok")]
  if (!length(ok)) {
    if (verbose) cat("  both variants failed — returning diagnostics, no fit\n")
    return(invisible(list(cma = cma_label, age = age_label, status = "failed", diag = results)))
  }

  qa <- sapply(ok, function(v) results[[v]]$qaic)
  winner_var <- ok[which.min(qa)]
  wfit <- results[[winner_var]]$fit
  cb_idx <- grep("^cb", names(coef(wfit)))
  stopifnot(length(cb_idx) == 25)

  list(
    cma             = cma_label,
    age             = age_label,
    status          = "ok",
    winner_variant  = winner_var,
    coef            = coef(wfit)[cb_idx],
    vcov            = vcov(wfit)[cb_idx, cb_idx],
    n_strata_A      = if (!is.null(results$A$n_strata)) results$A$n_strata else NA,
    n_strata_B      = if (!is.null(results$B$n_strata)) results$B$n_strata else NA,
    qaic_A          = if (!is.null(results$A$qaic)) results$A$qaic else Inf,
    qaic_B          = if (!is.null(results$B$qaic)) results$B$qaic else Inf,
    mean_dps_winner = results[[winner_var]]$mean_deaths_per_stratum,
    cb_template     = cb_template
  )
}

b <- paste(deparse(body(fit_stage1)), collapse = " ")
cat("binds cb:", grepl("cb <- cb_template", b, fixed = TRUE), "\n")
cat("three-outcome intact:", grepl("$converged", b, fixed = TRUE) & grepl("gnm raised", b, fixed = TRUE), "\n")
cat("returns cb_template:", grepl("cb_template = cb_template", b, fixed = TRUE), "\n")

binds cb: TRUE 
three-outcome intact: TRUE 
returns cb_template: TRUE 


cb <- cb_template bound in the frame the formula is built in

rerun mtl through run_city with the patched fit_stage1

predict mu [1.394, 0.13, 0.251] and deaths 203,988 reproduce exactly, both variants converge, winner A, theta 5, V 5x5 pos-def, mem flat

In [ ]:
mtl_out <- run_city("montreal_daymet_2015_2019.csv",
                    cma_age_data_mtlvan$montreal,
                    "Montreal", seed_offset = 24, L = L)

cat("\nstatus:", mtl_out$status, "\n")
print(mtl_out$diag)
cat("theta:", paste(round(mtl_out$red$theta_star, 3), collapse=" "), "\n")
cat("V diag:", paste(round(diag(mtl_out$red$V_star), 3), collapse=" "), "\n")
cat("V pos-def:", tryCatch({ chol(mtl_out$red$V_star); TRUE }, error = function(e) FALSE), "\n")
cat("Z:", paste(dim(mtl_out$Z), collapse=" x "), " da_ids:", length(mtl_out$da_ids), "\n")
cat("mem gb:", round(sum(gc()[,2])/1024, 2), "\n")


===== Montreal =====
  n_da 6504  mu [1.394, 0.13, 0.251]  temp NA 0
  deaths 203,988  lambda NA/Inf 0/0  pct zero 99.17

=== Stage 1: Montreal, age age_75_84 ===
  Variant A: converged, qAIC = 27967.1, strata = 3750
  Variant B: converged, qAIC = 65651.8, strata = 26250

status: ok 
        cma  n_da      mu1       mu2       mu3 deaths sliver_deaths ref_temp
     <char> <int>    <num>     <num>     <num>  <int>         <int>    <num>
1: Montreal  6504 1.394385 0.1301863 0.2509156 203988          1524 19.00753
   winner   qaic_A  qaic_B
   <char>    <num>   <num>
1:      A 27967.07 65651.8
theta: 5.818 -0.489 0.189 -2.868 7.717 
V diag: 7.784 4.115 5.631 4.622 10.692 
V pos-def: TRUE 
Z: 6504 x 17  da_ids: 6504 
mem gb: 1.3 



===== Montreal =====
  n_da 6504  mu [1.394, 0.13, 0.251]  temp NA 0
  deaths 203,988  lambda NA/Inf 0/0  pct zero 99.17

=== Stage 1: Montreal, age age_75_84 ===
  Variant A: converged, qAIC = 27967.1, strata = 3750
  Variant B: converged, qAIC = 65651.8, strata = 26250

status: ok
        cma  n_da      mu1       mu2       mu3 deaths sliver_deaths ref_temp
     <char> <int>    <num>     <num>     <num>  <int>         <int>    <num>
1: Montreal  6504 1.394385 0.1301863 0.2509156 203988          1524 19.00753
   winner   qaic_A  qaic_B
   <char>    <num>   <num>
1:      A 27967.07 65651.8
theta: 5.818 -0.489 0.189 -2.868 7.717
V diag: 7.784 4.115 5.631 4.622 10.692
V pos-def: TRUE
Z: 6504 x 17  da_ids: 6504
mem gb: 1.3

---

mtl through. mu [1.394, 0.13, 0.251] and deaths 203,988 reproduce.

A 27967 B 65652. B splits into 26250 strata against A's 3750 and the sliver holds 1524 deaths,
so B runs ~0.06 deaths/stratum, most strata are empty, contributing nothing, still charged for by qAIC.
A absorbs dow as covariate and holds 0.41. sliver artifact; B may win at full scale.

ref_temp 19.0 vs tor 19.4— each centred at its own sliver median, same percentile different
temperature, which is what makes the 5-vectors poolable

V diag 5th 10.7 vs tor's 37.4, both the top b-spline term above the 90th-pct knot where data
is thin, mtl's much tamer

mem 1.3gb after exit, sim released

check what OTT/CAL/QC need before looping.

ottawa is bi-provincial 24+35 so it needs ontario and quebec both

In [ ]:
f <- list.files(DRIVE)
cat("files:", length(f), "\n")
print(f)

cat("\n--- anything population-shaped ---\n")
print(grep("age|profile|pop|cma|census", f, ignore.case = TRUE, value = TRUE))

files: 28 
 [1] "calgary_da_shp.zip"             "calgary_daymet_2015_2019.csv"  
 [3] "Figures"                        "fns.R"                         
 [5] "mmt_da_mtl_2026-06-24.rds"      "montreal_da_shp.zip"           
 [7] "montreal_daymet_2015_2019.csv"  "mtl_da_long_2026-06-24.rds"    
 [9] "mtl_sim_2026-06-24.rds"         "Notebooks"                     
[11] "ottawa_da_shp.zip"              "ottawa_daymet_2015_2019.csv"   
[13] "quebec_da_shp.zip"              "quebec_daymet_2015_2019.csv"   
[15] "r_library.tar.gz"               "README.md"                     
[17] "saves_2026-05-26.tar.gz"        "saves_eod_2026-06-10.tar.gz"   
[19] "saves_eod_2026-06-17.tar.gz"    "saves_eod_2026-06-25.tar.gz"   
[21] "saves_mtlvan_clean.tar.gz"      "saves_pilot_2026-05-27.tar.gz" 
[23] "saves_pilot_2026-06-03.tar.gz"  "toronto_da_shp.zip"            
[25] "toronto_da.geojson"             "toronto_daymet_2015_2019.csv"  
[27] "vancouver_da_shp.zip"           "vancouver_daymet_2015_2019.

van through run_city. population already banked in cma_age_data_mtlvan, no download needed
ott/cal/qc need census profiles + dgrf + national shapefile, ~20-30 min of downloading, park
that until van is through

predict mu distinct from tor and mtl, n_da 3573,
deaths 60-200k vs v1's 55265 depending on mu2, ref_temp ~17 maritime, winner A

In [ ]:
van_out <- run_city("vancouver_daymet_2015_2019.csv",
                    cma_age_data_mtlvan$vancouver,
                    "Vancouver", seed_offset = 59, L = L)

cat("\nstatus:", van_out$status, "\n")
print(van_out$diag)
cat("theta:", paste(round(van_out$red$theta_star, 3), collapse=" "), "\n")
cat("V diag:", paste(round(diag(van_out$red$V_star), 3), collapse=" "), "\n")
cat("V pos-def:", tryCatch({ chol(van_out$red$V_star); TRUE }, error = function(e) FALSE), "\n")
cat("mem gb:", round(sum(gc()[,2])/1024, 2), "\n")


===== Vancouver =====
  n_da 3573  mu [-0.196, 0.331, -0.405]  temp NA 0
  deaths 56,157  lambda NA/Inf 0/0  pct zero 99.49

=== Stage 1: Vancouver, age age_75_84 ===
  Variant A: converged, qAIC = 27441.6, strata = 3750
  Variant B: converged, qAIC = 74910.6, strata = 26250

status: ok 
         cma  n_da        mu1       mu2        mu3 deaths sliver_deaths
      <char> <int>      <num>     <num>      <num>  <int>         <int>
1: Vancouver  3573 -0.1956219 0.3314771 -0.4049663  56157           649
   ref_temp winner   qaic_A   qaic_B
      <num> <char>    <num>    <num>
1: 17.04646      A 27441.62 74910.56
theta: -2.237 -5.594 -4.023 -4.792 -0.714 
V diag: 23.245 16.812 19.167 16.912 36.842 
V pos-def: TRUE 
mem gb: 1.32 


===== Vancouver =====
  n_da 3573  mu [-0.196, 0.331, -0.405]  temp NA 0
  deaths 56,157  lambda NA/Inf 0/0  pct zero 99.49

=== Stage 1: Vancouver, age age_75_84 ===
  Variant A: converged, qAIC = 27441.6, strata = 3750
  Variant B: converged, qAIC = 74910.6, strata = 26250

status: ok
         cma  n_da        mu1       mu2        mu3 deaths sliver_deaths
      <char> <int>      <num>     <num>      <num>  <int>         <int>
1: Vancouver  3573 -0.1956219 0.3314771 -0.4049663  56157           649
   ref_temp winner   qaic_A   qaic_B
      <num> <char>    <num>    <num>
1: 17.04646      A 27441.62 74910.56
theta: -2.237 -5.594 -4.023 -4.792 -0.714
V diag: 23.245 16.812 19.167 16.912 36.842
V pos-def: TRUE
mem gb: 1.32

---

van through. mu [-0.196, 0.331, -0.405] distinct from tor and mtl, n_da 3573, ref_temp 17.0
reproduces v1. maritime, 2.4°C below toronto

strata 3750/26250 identical across all three cities, fixed by sliver design 150 DA x 5yr x 5mo
under A, x7 dow under B, not a city property

deaths 56157 vs v1 55265 = 1.02x flat, where tor was 3.2x and mtl 1.43x. mu2 ordering
(mtl 0.13, van 0.331, tor 0.655) doesn't predict the multiplier — van maritime so fewer days
below its own mmt, less surface for exp(0.3·F2·cold) to act on, plausible.

V diag 16-37 vs mtl's 4-11, ~4x wider. 649 sliver deaths vs mtl's 1524, less information per
parameter. mixmeta inverse-weights so van contributes less to the pool

toronto through run_city.  ran hand-assembled at top level on 07-15, now through the wrapper
for Z + da_ids in the same shape as mtl/van, and as a reproduction check on run_city itself

da_age already in env from pilot_session

TOR is the biggest sim at 23.5M rows, so this is the slowest run tdy

predict every number reproduces 07-15: mu [-0.33, 0.655, 0.384] read first, deaths 610216,
n_da 7682, ref_temp 19.4, qaic A 27047 B 55302, theta [-0.843, -6.486, -5.099, -6.053, -3.616]
drift means run_city differs from the hand-assembled path

In [ ]:
tor_out <- run_city("toronto_daymet_2015_2019.csv",
                    da_age,
                    "Toronto", seed_offset = 35, L = L)

cat("\nstatus:", tor_out$status, "\n")
print(tor_out$diag)
cat("theta:", paste(round(tor_out$red$theta_star, 3), collapse=" "), "\n")
cat("V diag:", paste(round(diag(tor_out$red$V_star), 3), collapse=" "), "\n")
cat("V pos-def:", tryCatch({ chol(tor_out$red$V_star); TRUE }, error = function(e) FALSE), "\n")
cat("mem gb:", round(sum(gc()[,2])/1024, 2), "\n")


===== Toronto =====
  n_da 7682  mu [-0.33, 0.655, 0.384]  temp NA 0
  deaths 610,216  lambda NA/Inf 0/0  pct zero 98.83

=== Stage 1: Toronto, age age_75_84 ===
  Variant A: converged, qAIC = 27047.0, strata = 3750
  Variant B: converged, qAIC = 55301.9, strata = 26250

status: ok 
       cma  n_da        mu1       mu2       mu3 deaths sliver_deaths ref_temp
    <char> <int>      <num>     <num>     <num>  <int>         <int>    <num>
1: Toronto  7682 -0.3297828 0.6546325 0.3838666 610216          2419 19.38333
   winner   qaic_A   qaic_B
   <char>    <num>    <num>
1:      A 27047.04 55301.89
theta: -0.843 -6.486 -5.099 -6.053 -3.616 
V diag: 3.985 1.899 2.904 4.095 37.448 
V pos-def: TRUE 
mem gb: 1.34 


===== Toronto =====
  n_da 7682  mu [-0.33, 0.655, 0.384]  temp NA 0
  deaths 610,216  lambda NA/Inf 0/0  pct zero 98.83

=== Stage 1: Toronto, age age_75_84 ===
  Variant A: converged, qAIC = 27047.0, strata = 3750
  Variant B: converged, qAIC = 55301.9, strata = 26250

status: ok
       cma  n_da        mu1       mu2       mu3 deaths sliver_deaths ref_temp
    <char> <int>      <num>     <num>     <num>  <int>         <int>    <num>
1: Toronto  7682 -0.3297828 0.6546325 0.3838666 610216          2419 19.38333
   winner   qaic_A   qaic_B
   <char>    <num>    <num>
1:      A 27047.04 55301.89
theta: -0.843 -6.486 -5.099 -6.053 -3.616
V diag: 3.985 1.899 2.904 4.095 37.448
V pos-def: TRUE
mem gb: 1.34

---

toronto through run_city, reproduced and all exact to 07-15.

run_city is the same computation as the hand-assembled path.

mem 1.34gb after the largest sim today, 23.5M rows released on exit

confirm three reduced curves, sim regenerates at seed 42 but that's
43M rows of it, so the curves are worth storing



In [ ]:
red3 <- list(tor = tor_out$red, mtl = mtl_out$red, van = van_out$red)
diag3 <- rbindlist(list(tor_out$diag, mtl_out$diag, van_out$diag))
Z3 <- list(tor = tor_out$Z, mtl = mtl_out$Z, van = van_out$Z)
ids3 <- list(tor = tor_out$da_ids, mtl = mtl_out$da_ids, van = van_out$da_ids)

save_to_drive(list(red3_v2 = red3, diag3_v2 = diag3, Z3_v2 = Z3, ids3_v2 = ids3))

chk <- readRDS("/content/saves_eod/red3_v2.rds")
cat("read-back cities:", length(chk), " names:", paste(names(chk), collapse=" "), "\n")
cat("theta lengths:", paste(sapply(chk, function(x) length(x$theta_star)), collapse=" "), "\n")
print(diag3[, .(cma, n_da, deaths, sliver_deaths, ref_temp, qaic_A)])

saved 4 objects -> /content/drive/MyDrive/thesis/dlnm-pilot/saves_eod_2026-07-22.tar.gz 
read-back cities: 3  names: tor mtl van 
theta lengths: 5 5 5 
         cma  n_da deaths sliver_deaths ref_temp   qaic_A
      <char> <int>  <int>         <int>    <num>    <num>
1:   Toronto  7682 610216          2419 19.38333 27047.04
2:  Montreal  6504 203988          1524 19.00753 27967.07
3: Vancouver  3573  56157           649 17.04646 27441.62


saved 4 objects -> /content/drive/MyDrive/thesis/dlnm-pilot/saves_eod_2026-07-22.tar.gz
read-back cities: 3  names: tor mtl van
theta lengths: 5 5 5
         cma  n_da deaths sliver_deaths ref_temp   qaic_A
      <char> <int>  <int>         <int>    <num>    <num>
1:   Toronto  7682 610216          2419 19.38333 27047.04
2:  Montreal  6504 203988          1524 19.00753 27967.07
3: Vancouver  3573  56157           649 17.04646 27441.62

---
4 objects saved to saves_eod_2026-07-22, read-back 3 cities all 5-vectors

deaths 610216 / 203988 / 56157 across 7682 / 6504 / 3573 DAs — 10.9x death range on a 2.2x
DA range, so deaths track population not DA count. sliver deaths 2419 / 1524 / 649 same
ordering, which is what drives van's wider V_star

qaic_A 27047 / 27967 / 27442, tight band.


stage2 at K=3 on v2 curves. three 5-vectors stack to a 3x5 response, three 5x5 V_stars as
the S list, random intercept per cma. wrapper drops PCs (needs n_cma>4) and age_band (needs
>1), so intercept-only

the risk is reml on a 15-param Ψ from 3 groups with van's V_star 4x wider than mtl's

predict method reml read first (fixed = no between-city variance component), cholesky TRUE,
pooled vector closer to toronto than vancouver since inverse-vcov weighting and tor's V diag
2-4 vs van's 16-37

In [ ]:
reduced_list_v2 <- list(tor_out$red, mtl_out$red, van_out$red)
s2_v2 <- fit_stage2(reduced_list_v2)

stage2_v2 <- s2_v2$fit
chol_ok <- tryCatch({ chol(stage2_v2$Psi); TRUE }, error = function(e) FALSE)

cat("\nmethod:", s2_v2$method, " converged:", isTRUE(stage2_v2$converged), "\n")
cat("cholesky:", chol_ok, "\n")
cat("pooled:", paste(round(coef(stage2_v2), 3), collapse=" "), "\n")
cat("v1 pooled: 0.945 -1.986 -0.081 -4.688 7.110\n")
cat("df.residual:", stage2_v2$df.residual, "\n")
cat("\nper-city theta:\n")
print(round(s2_v2$theta_mat, 2))

  formula: cbind(theta1, theta2, theta3, theta4, theta5) ~ 1 | random: TRUE | method: reml

method: reml  converged: TRUE 
cholesky: FALSE 
pooled: 0.442 -4.614 -3.403 -5.038 1.07 
v1 pooled: 0.945 -1.986 -0.081 -4.688 7.110
df.residual: -5 

per-city theta:
     theta1 theta2 theta3 theta4 theta5
[1,]  -0.84  -6.49  -5.10  -6.05  -3.62
[2,]   5.82  -0.49   0.19  -2.87   7.72
[3,]  -2.24  -5.59  -4.02  -4.79  -0.71


formula: cbind(theta1, theta2, theta3, theta4, theta5) ~ 1 | random: TRUE | method: reml

method: reml  converged: TRUE
cholesky: FALSE
pooled: 0.442 -4.614 -3.403 -5.038 1.07
v1 pooled: 0.945 -1.986 -0.081 -4.688 7.110
df.residual: -5

per-city theta:
     theta1 theta2 theta3 theta4 theta5
[1,]  -0.84  -6.49  -5.10  -6.05  -3.62
[2,]   5.82  -0.49   0.19  -2.87   7.72
[3,]  -2.24  -5.59  -4.02  -4.79  -0.71

---

reml converged but cholesky FALSE, Ψ not positive-definite. 15 free params in a 5x5 Ψ from
3 groups, so reml returns an estimate sitting on the boundary of the parameter space.

v1 got cholesky TRUE at the same K and shape, so the difference is the inputs. per-city
theta1: tor -0.84, mtl +5.82, van -2.24; v2 mu offsets produced between-city spread
and three points can't support it across 15 params.

pooled [0.442, -4.614, -3.403, -5.038, 1.07] sits nearer toronto than vancouver, matching
inverse-vcov weighting: tor V diag 2-4, van 16-37

df.residual -5 as expected at K=3, clears at K=6

at K=6 PCs enter and absorb some between-city spread into fixed effects instead of Ψ.

download inputs for ott/cal/qc. dgrf + national shapefile for the DA lists, then ontario,
quebec, alberta profiles. ottawa is bi-provincial 24+35 so it needs ontario and quebec both

In [ ]:
dir.create("/content/statcan", showWarnings = FALSE)
options(timeout = 3600)

da_zip_url <- "https://www12.statcan.gc.ca/census-recensement/2021/geo/sip-pis/boundary-limites/files-fichiers/lda_000b21a_e.zip"
dgrf_url   <- "https://www12.statcan.gc.ca/census-recensement/2021/geo/sip-pis/dguid-idugd/files-fichiers/2021_98260004.zip"
base_url   <- "https://www12.statcan.gc.ca/census-recensement/2021/dp-pd/prof/details/download-telecharger/comp/GetFile.cfm?Lang=E&FILETYPE=CSV&GEONO="

geono <- c(ontario = "006_Ontario", quebec = "006_Quebec", alberta = "006_Alberta")

if (!file.exists("/content/statcan/da_boundaries.zip"))
  system(sprintf('wget -q -O /content/statcan/da_boundaries.zip "%s"', da_zip_url))
if (!file.exists("/content/statcan/dgrf.zip"))
  system(sprintf('wget -q -O /content/statcan/dgrf.zip "%s"', dgrf_url))

for (p in names(geono)) {
  dest <- sprintf("/content/statcan/profile_%s.zip", p)
  if (!file.exists(dest))
    system(sprintf('wget -q -O "%s" "%s%s"', dest, base_url, geono[p]))
}

f <- list.files("/content/statcan", full.names = TRUE)
cat("files:", length(f), "\n")
for (x in f) cat(sprintf("%-45s %8.1f MB\n", basename(x), file.size(x)/1e6))

files: 5 
da_boundaries.zip                                197.0 MB
dgrf.zip                                           2.2 MB
profile_alberta.zip                                0.0 MB
profile_ontario.zip                              757.0 MB
profile_quebec.zip                               540.8 MB


files: 5
da_boundaries.zip                                197.0 MB
dgrf.zip                                           2.2 MB
profile_alberta.zip                                0.0 MB
profile_ontario.zip                              757.0 MB
profile_quebec.zip                               540.8 MB

---

alberta 0.0 mb, ontario and quebec fine, so wget and the url pattern work and alberta's geono
code is wrong.

§9.2 I logged BC as 006_BC_CB not 006_BritishColumbia so the codes aren't
uniformly the spelled-out name

In [ ]:
system('wget -S -O /content/statcan/ab_test.zip "https://www12.statcan.gc.ca/census-recensement/2021/dp-pd/prof/details/download-telecharger/comp/GetFile.cfm?Lang=E&FILETYPE=CSV&GEONO=006_Alberta" 2>&1 | head -20')

cat("\nsize:", file.size("/content/statcan/ab_test.zip"), "bytes\n")
cat("\n--- first bytes ---\n")
system('head -c 400 /content/statcan/ab_test.zip')

for (code in c("006_AB", "006_Alberta_AB", "006_ALBERTA")) {
  dest <- sprintf("/content/statcan/ab_%s.zip", code)
  system(sprintf('wget -q -O "%s" "https://www12.statcan.gc.ca/census-recensement/2021/dp-pd/prof/details/download-telecharger/comp/GetFile.cfm?Lang=E&FILETYPE=CSV&GEONO=%s"', dest, code))
  cat(sprintf("%-16s %8.1f MB\n", code, file.size(dest)/1e6))
}


size: 4099 bytes

--- first bytes ---
006_AB                0.0 MB
006_Alberta_AB        0.0 MB
006_ALBERTA           0.0 MB


size: 4099 bytes

--- first bytes ---
006_AB                0.0 MB
006_Alberta_AB        0.0 MB
006_ALBERTA           0.0 MB

system() isn't returning stdout so the header read came back blank.

read the failed response
in R instead


In [ ]:
x <- readLines("/content/statcan/ab_test.zip", warn = FALSE, n = 60)
cat(paste(x, collapse = "\n"))

<!DOCTYPE html><!--[if lt IE 9]><html class="no-js lt-ie9" lang="en" dir="ltr"><![endif]--><!--[if gt IE 8]><!-->
<html class="no-js" lang="en" dir="ltr">
<!--<![endif]-->
<head>
<meta charset="utf-8">
<!-- Web Experience Toolkit (WET) / Boîte à outils de l'expérience Web (BOEW)
		wet-boew.github.io/wet-boew/License-en.html / wet-boew.github.io/wet-boew/Licence-fr.html -->
<title>File not found | Fichier non trouv&eacute;</title>
<meta content="width=device-width,initial-scale=1" name="viewport">
<!-- Meta data -->
<meta name="robots" content="noindex, nofollow, noarchive">
<!-- Meta data-->
<!--[if gte IE 9 | !IE ]><!-->
<link href="/wet-boew4b/assets/favicon.ico" rel="icon" type="image/x-icon">
<link rel="stylesheet" href="/wet-boew4b/css/wet-boew.min.css">
<!--<![endif]-->
<link rel="stylesheet" href="/wet-boew4b/css/theme-srv.min.css">
<!--[if lt IE 9]>
<link href="/wet-boew4b/assets/favicon.ico" rel="shortcut icon" />
<link rel="stylesheet" href="/wet-boew4b/css/ie8-wet-boew.min.c

<!DOCTYPE html><!--[if lt IE 9]><html class="no-js lt-ie9" lang="en" dir="ltr"><![endif]--><!--[if gt IE 8]><!-->
<html class="no-js" lang="en" dir="ltr">
<!--<![endif]-->
<head>
<meta charset="utf-8">
<!-- Web Experience Toolkit (WET) / Boîte à outils de l'expérience Web (BOEW)
		wet-boew.github.io/wet-boew/License-en.html / wet-boew.github.io/wet-boew/Licence-fr.html -->
<title>File not found | Fichier non trouv&eacute;</title>
<meta content="width=device-width,initial-scale=1" name="viewport">
<!-- Meta data -->
<meta name="robots" content="noindex, nofollow, noarchive">
<!-- Meta data-->
<!--[if gte IE 9 | !IE ]><!-->
<link href="/wet-boew4b/assets/favicon.ico" rel="icon" type="image/x-icon">
<link rel="stylesheet" href="/wet-boew4b/css/wet-boew.min.css">
<!--<![endif]-->
<link rel="stylesheet" href="/wet-boew4b/css/theme-srv.min.css">
<!--[if lt IE 9]>
<link href="/wet-boew4b/assets/favicon.ico" rel="shortcut icon" />
<link rel="stylesheet" href="/wet-boew4b/css/ie8-wet-boew.min.css" />
<script src="http://ajax.googleapis.com/ajax/libs/jquery/1.11.0/jquery.min.js"></script>
<script src="/wet-boew4b/js/ie8-wet-boew.min.js"></script>
<![endif]-->
<noscript><link rel="stylesheet" href="/wet-boew4b/css/noscript.min.css" /></noscript>
</head>
<body vocab="http://schema.org/" typeof="WebPage">
<header role="banner" id="wb-bnr" class="container">
<div class="row">
<div class="col-sm-6">
<object id="gcwu-sig" type="image/svg+xml" tabindex="-1" role="img" data="/wet-boew4b/assets/sig-blk-en.svg" aria-label="Government of Canada"></object>
</div>
<div class="col-sm-6">
<object id="wmms" type="image/svg+xml" tabindex="-1" role="img" data="/wet-boew4b/assets/wmms-blk.svg" aria-label="Symbol of the Government of Canada"></object>
</div>
</div>
</header>
<main role="main" property="mainContentOfPage" class="container">
<div class="row mrgn-tp-lg">
<h1 class="wb-inv">File not found / <span lang="fr">Fichier non trouvé</span></h1>
<section class="col-md-6">
<h2><span class="glyphicon glyphicon-warning-sign mrgn-rght-md"></span> File not found</h2>
<p>Sorry, the web page you requested cannot be found.</p>
    <p>This may have happened for a number of different reasons:</p>
    <ul>
    	<li>The page is temporarily offline for updating</li>
        <li>You may have typed the URL incorrectly into the address bar of your browser</li>
        <li>The page has been moved or removed from our site</li>
    </ul>
<p>Return to the <a href="http://www12.statcan.gc.ca/census-recensement/index-eng.cfm">Census Program</a> or <a href="http://www.statcan.gc.ca/start-debut-eng.html">Statistics Canada home page</a> or <a href="http://www.statcan.gc.ca/reference/refcentre-centreref/index-eng.htm">contact us</a></p>
</section>
<section class="col-md-6" lang="fr">
<h2><span class="glyphicon glyphicon-warning-sign mrgn-rght-md"></span> Fichier introuvable</h2>
<p>D&eacute;sol&eacute;, la page Web demand&eacute;e est introuvable.</p>
<p>Plusieurs raisons diff&eacute;rentes peuvent en &ecirc;tre la cause :</p>
<ul>
	<li>La page est temporairement hors ligne en vue d'une mise &agrave; jour</li>
    <li>Vous avez entr&eacute; incorrectement l'adresse URL dans la barre d'adresse de votre fureteur</li>
    <li>La page a &eacute;t&eacute; d&eacute;plac&eacute;e ou retir&eacute;e de notre site</li>
</ul>
<p>Retournez à <a href="http://www12.statcan.gc.ca/census-recensement/index-fra.cfm">Programme du recensement</a> ou <a href="http://www.statcan.gc.ca/start-debut-fra.html">la page d'accueil de Statistique Canada</a> ou <a href="http://www.statcan.gc.ca/reference/refcentre-centreref/index-fra.htm">contactez-nous</a></p>

---

Generic error.

Ottawa and Québec now, Calgary if there's time.

unzip shapefile + dgrf, read both. shapefile has no cma code so dgrf supplies DA → CMA

predict da_all 57932 rows, dgrf 498786 x 16 — the 2.2mb zip was small so this is
the real check on it

In [ ]:
if (!dir.exists("/content/statcan/da_extracted")) {
  dir.create("/content/statcan/da_extracted")
  unzip("/content/statcan/da_boundaries.zip", exdir = "/content/statcan/da_extracted")
}
if (!dir.exists("/content/statcan/dgrf_extracted")) {
  dir.create("/content/statcan/dgrf_extracted")
  unzip("/content/statcan/dgrf.zip", exdir = "/content/statcan/dgrf_extracted")
}

cat("da_extracted:", paste(list.files("/content/statcan/da_extracted"), collapse=" "), "\n")
cat("dgrf_extracted:", paste(list.files("/content/statcan/dgrf_extracted"), collapse=" "), "\n\n")

da_all <- sf::st_read("/content/statcan/da_extracted/lda_000b21a_e.shp", quiet = TRUE)
da_all$DAUID <- as.character(da_all$DAUID)
cat("da_all rows:", nrow(da_all), " (expect 57932)\n")
cat("cols:", paste(colnames(da_all), collapse=" "), "\n")

dgrf <- data.table::fread("/content/statcan/dgrf_extracted/2021_98260004.csv")
cat("\ndgrf rows:", nrow(dgrf), " cols:", ncol(dgrf), " (expect 498786 x 16)\n")
cat("has CMADGUID:", "CMADGUID_RMRIDUGD" %in% names(dgrf),
    " has DADGUID:", "DADGUID_ADIDUGD" %in% names(dgrf), "\n")

da_extracted: lda_000b21a_e.dbf lda_000b21a_e.prj lda_000b21a_e.shp lda_000b21a_e.shx lda_000b21a_e.xml 
dgrf_extracted: 2021_98260004.csv 

da_all rows: 57932  (expect 57932)
cols: DAUID DGUID LANDAREA PRUID geometry 

dgrf rows: 498786  cols: 16  (expect 498786 x 16)
has CMADGUID: TRUE  has DADGUID: TRUE 


da_all 57932 rows, cols DAUID DGUID LANDAREA PRUID geometry, no cma code which is why dgrf
is needed. dgrf 498786 x 16, both CMADGUID and DADGUID present

DA lists for ottawa and québec. filter dgrf to each cma dguid, dedup DB → DA, strip dauid
from dguid. ottawa 2021S0503505, québec 2021S0503421

predict ottawa 2045 DAs prefixes 24 and 35 read first (bi-provincial; both banks of the river
means the dguid filter caught it), québec 1317 prefix 24, missing 0 both

In [ ]:
new_dguids <- c(Ottawa = "2021S0503505", Quebec = "2021S0503421")

new_da_lists <- lapply(new_dguids, function(dg) {
  db_rows <- dgrf[CMADGUID_RMRIDUGD == dg]
  substr(unique(db_rows$DADGUID_ADIDUGD), 10, 17)
})

for (nm in names(new_da_lists)) {
  d <- new_da_lists[[nm]]
  poly <- da_all[da_all$DAUID %in% d, ]
  cat(sprintf("%-8s DAs: %5d  in shapefile: %5d  missing: %d  prefixes: %s\n",
              nm, length(d), nrow(poly), length(setdiff(d, poly$DAUID)),
              paste(sort(unique(substr(d, 1, 2))), collapse=",")))
}

Ottawa   DAs:  2045  in shapefile:  2045  missing: 0  prefixes: 24,35
Quebec   DAs:  1317  in shapefile:  1317  missing: 0  prefixes: 24


Ottawa   DAs:  2045  in shapefile:  2045  missing: 0  prefixes: 24,35
Quebec   DAs:  1317  in shapefile:  1317  missing: 0  prefixes: 24

---

ottawa 2045 DAs prefixes 24,35 — dguid filter caught both banks. québec 1317 prefix 24.
missing 0 both, every dauid found a polygon in da_all. 9.7 reproduced

age tables for ottawa and québec.

unzip ontario + quebec profiles, grep each city's dauids
out, filter 18 age ids, aggregate to 4 bands. ottawa greps both provinces and stacks

predict ottawa 2045 DAs read first (fewer = grep missed or a file came up short), québec 1317,
pops ~1.49M and ~840k, band shares near toronto's 83% / 2%

In [ ]:
prof_zips <- c(ontario = "/content/statcan/profile_ontario.zip",
               quebec  = "/content/statcan/profile_quebec.zip")
prov_sfx  <- c(ontario = "Ontario", quebec = "Quebec")

for (p in names(prof_zips)) {
  ex <- sprintf("/content/statcan/profile_%s_extracted", p)
  if (!dir.exists(ex)) { dir.create(ex); unzip(prof_zips[p], exdir = ex) }
  cat(p, ":", paste(list.files(ex), collapse=" "), "\n")
}

age_ids  <- c(10,11,12,14,15,16,17,18,19,20,21,22,23, 25,26, 27,28, 29)
band_map <- data.table(CHARACTERISTIC_ID = age_ids,
  band = c(rep("age_0_64",13), rep("age_65_74",2), rep("age_75_84",2), rep("age_85p",1)))

city_provs <- list(Ottawa = c("ontario","quebec"), Quebec = "quebec")
new_age <- list()

for (city in names(city_provs)) {
  dauids <- new_da_lists[[city]]
  pat <- sprintf("/content/statcan/%s_patterns.txt", tolower(city))
  writeLines(paste0('"', dauids, '"'), pat)

  parts <- list()
  for (p in city_provs[[city]]) {
    src <- sprintf("/content/statcan/profile_%s_extracted/98-401-X2021006_English_CSV_data_%s.csv",
                   p, prov_sfx[p])
    out <- sprintf("/content/statcan/profile_%s_%s.csv", tolower(city), p)
    if (!file.exists(out)) {
      system(sprintf('head -1 "%s" > "%s"', src, out))
      system(sprintf('grep -F -f "%s" "%s" >> "%s"', pat, src, out))
    }
    parts[[p]] <- fread(out, select = c("ALT_GEO_CODE","CHARACTERISTIC_ID","C1_COUNT_TOTAL"))
  }
  prof <- rbindlist(parts)
  prof <- merge(prof[CHARACTERISTIC_ID %in% age_ids], band_map, by = "CHARACTERISTIC_ID")
  prof[is.na(C1_COUNT_TOTAL), C1_COUNT_TOTAL := 0]

  dw <- dcast(prof[, .(pop = sum(C1_COUNT_TOTAL)), by = .(ALT_GEO_CODE, band)],
              ALT_GEO_CODE ~ band, value.var = "pop")
  dw[, total := age_0_64 + age_65_74 + age_75_84 + age_85p]
  dw[, ALT_GEO_CODE := as.character(ALT_GEO_CODE)]
  new_age[[city]] <- dw

  cat(sprintf("\n%-8s DAs: %5d / %5d  pop: %s\n", city, nrow(dw), length(dauids),
              format(sum(dw$total), big.mark=",")))
  cat(sprintf("  bands: 0-64 %.1f%%  65-74 %.1f%%  75-84 %.1f%%  85+ %.1f%%\n",
              100*sum(dw$age_0_64)/sum(dw$total), 100*sum(dw$age_65_74)/sum(dw$total),
              100*sum(dw$age_75_84)/sum(dw$total), 100*sum(dw$age_85p)/sum(dw$total)))
}

ontario : 98-401-X2021006_English_CSV_data_Ontario.csv 98-401-X2021006_English_meta.txt 98-401-X2021006_Geo_starting_row_Ontario.CSV README_meta.txt 
quebec : 98-401-X2021006_English_CSV_data_Quebec.csv 98-401-X2021006_English_meta.txt 98-401-X2021006_Geo_starting_row_Quebec.CSV README_meta.txt 

Ottawa   DAs:  2045 /  2045  pop: 1,487,965
  bands: 0-64 83.0%  65-74 9.9%  75-84 5.0%  85+ 2.0%

Quebec   DAs:  1317 /  1317  pop: 839,300
  bands: 0-64 78.4%  65-74 12.3%  75-84 6.8%  85+ 2.5%


ontario : 98-401-X2021006_English_CSV_data_Ontario.csv 98-401-X2021006_English_meta.txt 98-401-X2021006_Geo_starting_row_Ontario.CSV README_meta.txt
quebec : 98-401-X2021006_English_CSV_data_Quebec.csv 98-401-X2021006_English_meta.txt 98-401-X2021006_Geo_starting_row_Quebec.CSV README_meta.txt

Ottawa   DAs:  2045 /  2045  pop: 1,487,965
  bands: 0-64 83.0%  65-74 9.9%  75-84 5.0%  85+ 2.0%

Quebec   DAs:  1317 /  1317  pop: 839,300
  bands: 0-64 78.4%  65-74 12.3%  75-84 6.8%  85+ 2.5%

---

ottawa 2045 DAs / 2045 in list, pop 1,487,965,bi-provincial grep stacked ontario + quebec
parts without losing a DA. québec 1317 / 1317, pop 839,300. both match published cma totals

bands: ottawa 83.0 / 9.9 / 5.0 / 2.0, near toronto's 83.8 / 9.2 / 4.9 / 2.1. québec city
older—78.4 under 65 and 6.8 in the 75-84 band we model, so proportionally more population
at risk

note: alberta has no DA-level profile file. the DA rows are per-region not per-province: atlantic,
quebec, ontario, prairies, BC, territories. alberta is inside prairies. 006_Ontario and
006_Quebec work because those provinces are their own regions, which reads like a province
naming convention and isn't

ottawa + québec through run_city. both smaller than vancouver so quick

seed note: ottawa is bi-provincial 24+35, and 35 is toronto's offset while 24 is québec's,
so ottawa gets 63 to stay distinct; departure from the province-prefix convention

predict mu distinct from all three banked cities read first, n_da ~2045 and ~1317 less
zero-pop, québec loses 1 water DA per §9.7, ref_temp near 19 both continental, winner A

Both cities run back to back in one block since run_city is proven and each releases its sim on exit. QC at 1317 DAs is the smallest sim today by a wide margin.

In [ ]:
ott_out <- run_city("ottawa_daymet_2015_2019.csv",
                    new_age$Ottawa, "Ottawa", seed_offset = 63, L = L)

qc_out  <- run_city("quebec_daymet_2015_2019.csv",
                    new_age$Quebec, "Quebec", seed_offset = 24, L = L)

for (o in list(ott_out, qc_out)) {
  cat("\nstatus:", o$status, "\n")
  print(o$diag)
  cat("theta:", paste(round(o$red$theta_star, 3), collapse=" "), "\n")
  cat("V diag:", paste(round(diag(o$red$V_star), 3), collapse=" "), "\n")
  cat("V pos-def:", tryCatch({ chol(o$red$V_star); TRUE }, error = function(e) FALSE), "\n")
}
cat("\nmem gb:", round(sum(gc()[,2])/1024, 2), "\n")


===== Ottawa =====
  n_da 2042  mu [-0.776, -0.28, 0.08]  temp NA 0
  deaths 51,476  lambda NA/Inf 0/0  pct zero 99.30

=== Stage 1: Ottawa, age age_75_84 ===
  Variant A: converged, qAIC = 28992.7, strata = 3750
  Variant B: converged, qAIC = 66496.5, strata = 26250

===== Quebec =====
  n_da 1310  mu [1.394, 0.13, 0.251]  temp NA 0
  deaths 39,207  lambda NA/Inf 0/0  pct zero 99.13

=== Stage 1: Quebec, age age_75_84 ===
  Variant A: converged, qAIC = 29662.9, strata = 3750
  Variant B: converged, qAIC = 72518.7, strata = 26250

status: ok 
      cma  n_da        mu1        mu2        mu3 deaths sliver_deaths ref_temp
   <char> <int>      <num>      <num>      <num>  <int>         <int>    <num>
1: Ottawa  2042 -0.7761144 -0.2796926 0.08028853  51476          1247 18.58291
   winner   qaic_A  qaic_B
   <char>    <num>   <num>
1:      A 28992.71 66496.5
theta: -3.84 -6.251 -7.395 -3.388 0.145 
V diag: 9.649 4.819 6.987 4.866 13.518 
V pos-def: TRUE 

status: ok 
      cma  n_da      

===== Ottawa =====
  n_da 2042  mu [-0.776, -0.28, 0.08]  temp NA 0
  deaths 51,476  lambda NA/Inf 0/0  pct zero 99.30

=== Stage 1: Ottawa, age age_75_84 ===
  Variant A: converged, qAIC = 28992.7, strata = 3750
  Variant B: converged, qAIC = 66496.5, strata = 26250

===== Quebec =====
  n_da 1310  mu [1.394, 0.13, 0.251]  temp NA 0
  deaths 39,207  lambda NA/Inf 0/0  pct zero 99.13

=== Stage 1: Quebec, age age_75_84 ===
  Variant A: converged, qAIC = 29662.9, strata = 3750
  Variant B: converged, qAIC = 72518.7, strata = 26250

status: ok
      cma  n_da        mu1        mu2        mu3 deaths sliver_deaths ref_temp
   <char> <int>      <num>      <num>      <num>  <int>         <int>    <num>
1: Ottawa  2042 -0.7761144 -0.2796926 0.08028853  51476          1247 18.58291
   winner   qaic_A  qaic_B
   <char>    <num>   <num>
1:      A 28992.71 66496.5
theta: -3.84 -6.251 -7.395 -3.388 0.145
V diag: 9.649 4.819 6.987 4.866 13.518
V pos-def: TRUE

status: ok
      cma  n_da      mu1       mu2       mu3 deaths sliver_deaths ref_temp
   <char> <int>    <num>     <num>     <num>  <int>         <int>    <num>
1: Quebec  1310 1.394385 0.1301863 0.2509156  39207          1374 16.79456
   winner   qaic_A  qaic_B
   <char>    <num>   <num>
1:      A 29662.87 72518.7
theta: -1.592 -2.925 -4.502 0.204 -10.074
V diag: 4.181 2.46 3.078 3.691 11.905
V pos-def: TRUE

mem gb: 2.02

---

ottawa n_da 2042 (3 zero-pop dropped, 0 water per §9.7), deaths 51,476, ref_temp 18.6.
québec n_da 1310 (7 dropped), deaths 39,207, ref_temp 16.8—coolest of the five, further north

both converged, A wins both, qaic_A 28993 and 29663, V pos-def both. ottawa's V diag 5-14,
québec's 2-12, both tighter than vancouver's 16-37 — sliver deaths 1247 and 1374 against
van's 649

Only problem québec mu [1.394, 0.13, 0.251] is montréal's, exact. This is seed collision; both got offset 24
because the convention is 42 + province prefix and both cities are in quebec. two of five
cities now sit at the same position in vulnerability space, which is the quantity stage 2
regresses on

the convention breaks whenever a province holds more than one CMA. at 41 CMAs across ten
provinces that's guaranteed

rerun québec with seed_offset 421, its cma code. unique by construction where the province
prefix wasn't — montréal and québec city both sit in province 24.

predict mu distinct from all four banked and specifically not montréal's

In [ ]:
qc_out <- run_city("quebec_daymet_2015_2019.csv",
                   new_age$Quebec, "Quebec", seed_offset = 421, L = L)

cat("\nstatus:", qc_out$status, "\n")
print(qc_out$diag)
cat("theta:", paste(round(qc_out$red$theta_star, 3), collapse=" "), "\n")
cat("V diag:", paste(round(diag(qc_out$red$V_star), 3), collapse=" "), "\n")
cat("V pos-def:", tryCatch({ chol(qc_out$red$V_star); TRUE }, error = function(e) FALSE), "\n")

mu_all <- rbind(tor_out$diag[, .(cma, mu1, mu2, mu3)], mtl_out$diag[, .(cma, mu1, mu2, mu3)],
                van_out$diag[, .(cma, mu1, mu2, mu3)], ott_out$diag[, .(cma, mu1, mu2, mu3)],
                qc_out$diag[, .(cma, mu1, mu2, mu3)])
cat("\n")
print(mu_all)
cat("\nduplicate mu rows:", sum(duplicated(mu_all[, .(mu1, mu2, mu3)])), " (must be 0)\n")


===== Quebec =====
  n_da 1310  mu [-0.077, -1.059, 0.2]  temp NA 0
  deaths 26,977  lambda NA/Inf 0/0  pct zero 99.34

=== Stage 1: Quebec, age age_75_84 ===
  Variant A: converged, qAIC = 28083.3, strata = 3750
  Variant B: converged, qAIC = 71150.7, strata = 26250

status: ok 
      cma  n_da         mu1       mu2       mu3 deaths sliver_deaths ref_temp
   <char> <int>       <num>     <num>     <num>  <int>         <int>    <num>
1: Quebec  1310 -0.07723611 -1.058581 0.2002045  26977           982 16.79456
   winner   qaic_A   qaic_B
   <char>    <num>    <num>
1:      A 28083.29 71150.71
theta: -3.304 -4.23 -6.111 -1.893 -9.115 
V diag: 5.144 3.079 3.737 4.259 12.836 
V pos-def: TRUE 

         cma         mu1        mu2         mu3
      <char>       <num>      <num>       <num>
1:   Toronto -0.32978280  0.6546325  0.38386660
2:  Montreal  1.39438480  0.1301863  0.25091561
3: Vancouver -0.19562189  0.3314771 -0.40496631
4:    Ottawa -0.77611436 -0.2796926  0.08028853
5:    Quebec

===== Quebec =====
  n_da 1310  mu [-0.077, -1.059, 0.2]  temp NA 0
  deaths 26,977  lambda NA/Inf 0/0  pct zero 99.34

=== Stage 1: Quebec, age age_75_84 ===
  Variant A: converged, qAIC = 28083.3, strata = 3750
  Variant B: converged, qAIC = 71150.7, strata = 26250

status: ok
      cma  n_da         mu1       mu2       mu3 deaths sliver_deaths ref_temp
   <char> <int>       <num>     <num>     <num>  <int>         <int>    <num>
1: Quebec  1310 -0.07723611 -1.058581 0.2002045  26977           982 16.79456
   winner   qaic_A   qaic_B
   <char>    <num>    <num>
1:      A 28083.29 71150.71
theta: -3.304 -4.23 -6.111 -1.893 -9.115
V diag: 5.144 3.079 3.737 4.259 12.836
V pos-def: TRUE

         cma         mu1        mu2         mu3
      <char>       <num>      <num>       <num>
1:   Toronto -0.32978280  0.6546325  0.38386660
2:  Montreal  1.39438480  0.1301863  0.25091561
3: Vancouver -0.19562189  0.3314771 -0.40496631
4:    Ottawa -0.77611436 -0.2796926  0.08028853
5:    Quebec -0.07723611 -1.0585815  0.20020454

duplicate mu rows: 0  (must be 0)

---

québec rerun at seed_offset 421. mu [-0.077, -1.059, 0.2], distinct from all four, duplicate
check 0

deaths 39,207 → 26,977. mu2 went +0.13 → -1.059 and the cold channel is exp(0.3·F2·cold),
so a negative offset shrinks where toronto's +0.655 inflated 3.2x

n_da 1310 and ref_temp 16.79456 unchanged to the digit

mu2 across five cities now spans -1.06 to +0.655

cma-mean vulnerability variables + pca. each city's Z is n_da x 17, column mean gives one
17-vector per city, 5x17 total. pca on that gives PC1-3 as stage 2 meta-regressors

the mu offsets are what make the rows differ — they shift the latents, which propagate
through L into Z. under v1 every row would sit near zero

note: 5 observations x 17 variables, so max 4 non-zero components and PC1-3 explains nearly
everything by arithmetic not by finding

predict Z_means 5x17 rows distinct read first, cum var >95% (not meaningful at this K),
scores 5x3

cma-mean vulnerability variables + pca. each city's Z is n_da x 17, column mean gives one
17-vector per city, 5x17 total. pca on that gives PC1-3 as stage 2 meta-regressors

the mu offsets are what make the rows differ; they shift the latents, which propagate
through L into Z.

note: 5 observations x 17 variables, so max 4 non-zero components and PC1-3 explains nearly
everything by arithmetic not by finding

predict Z_means 5x17 rows distinct read first, cum var >95% (not meaningful at this K),
scores 5x3

In [ ]:
Z_list <- list(Toronto = tor_out$Z, Montreal = mtl_out$Z, Vancouver = van_out$Z,
               Ottawa = ott_out$Z, Quebec = qc_out$Z)

Z_means <- t(sapply(Z_list, colMeans))
cat("Z_means:", paste(dim(Z_means), collapse=" x "), "\n")
cat("rows distinct:", nrow(unique(Z_means)) == nrow(Z_means), "\n")
print(round(Z_means[, 1:6], 3))

pca_cma <- prcomp(Z_means, scale. = TRUE)
cum_var <- summary(pca_cma)$importance["Cumulative Proportion", ]
cat("\ncum var PC1-3:", round(cum_var[3], 4), "\n")
cat("per-PC:", paste(round(summary(pca_cma)$importance["Proportion of Variance", 1:4], 3), collapse=" "), "\n")

cma_predictors <- data.table(CMA = rownames(Z_means),
                             PC1 = pca_cma$x[,1], PC2 = pca_cma$x[,2], PC3 = pca_cma$x[,3])
print(cma_predictors)

Z_means: 5 x 17 
rows distinct: TRUE 
          vuln01 vuln02 vuln03 vuln04 vuln05 vuln06
Toronto   -0.302 -0.231 -0.200  0.165 -0.227 -0.530
Montreal   1.249  0.970  0.830 -0.687  0.971 -0.090
Vancouver -0.164 -0.127 -0.118  0.094 -0.126 -0.268
Ottawa    -0.695 -0.536 -0.453  0.383 -0.547  0.224
Quebec    -0.099 -0.059 -0.065  0.057 -0.074  0.867

cum var PC1-3: 0.9999 
per-PC: 0.431 0.336 0.233 0 
         CMA       PC1         PC2        PC3
      <char>     <num>       <num>      <num>
1:   Toronto -1.998866 -0.71173777 -2.9258033
2:  Montreal -3.730649  0.95436513  1.8301977
3: Vancouver  1.354757 -3.38203234  1.6156235
4:    Ottawa  2.233846 -0.01748303 -1.0322194
5:    Quebec  2.140912  3.15688801  0.5122016


Z_means: 5 x 17
rows distinct: TRUE
          vuln01 vuln02 vuln03 vuln04 vuln05 vuln06
Toronto   -0.302 -0.231 -0.200  0.165 -0.227 -0.530
Montreal   1.249  0.970  0.830 -0.687  0.971 -0.090
Vancouver -0.164 -0.127 -0.118  0.094 -0.126 -0.268
Ottawa    -0.695 -0.536 -0.453  0.383 -0.547  0.224
Quebec    -0.099 -0.059 -0.065  0.057 -0.074  0.867

cum var PC1-3: 0.9999
per-PC: 0.431 0.336 0.233 0
         CMA       PC1         PC2        PC3
      <char>     <num>       <num>      <num>
1:   Toronto -1.998866 -0.71173777 -2.9258033
2:  Montreal -3.730649  0.95436513  1.8301977
3: Vancouver  1.354757 -3.38203234  1.6156235
4:    Ottawa  2.233846 -0.01748303 -1.0322194
5:    Quebec  2.140912  3.15688801  0.5122016

---

Z_means 5x17, all rows distinct. cum var PC1-3 0.9999 with PC4 exactly 0—5 observations
caps at 4 components; arithmetic not structure.

per-PC 0.431 / 0.336 / 0.233, flat, all three PCs carry
comparable weight as predictors. no dominant direction at K=5

rows track mu1: montréal +1.394 runs positive across vuln01-05, ottawa -0.776 runs negative.
vuln04 flips sign throughout, which is L[4,1] = -0.5, the one negative F1 loading

vuln06 breaks the F1 pattern — québec +0.867, ottawa +0.224 — because L[6,2] = -0.8 makes it
F2-driven, and québec's mu2 is -1.059

stage2 at K=5 with PC predictors. five 5-vectors stack to 5x5, five V_stars as S lis PC1t.

4 fixed-effect coefficients per outcome from 5 cities, so 1 residual df per outcome before Ψ

predict formula shows the PCs read first, method reml
or fixed, cholesky is the K=3 miss, df.residual positive where K=3 was -5,
PC slopes non-zero or the map stays flat

In [ ]:
reduced_list_5 <- list(tor_out$red, mtl_out$red, van_out$red, ott_out$red, qc_out$red)

s2_v2_k5 <- fit_stage2(reduced_list_5, cma_predictors_df = cma_predictors)
stage2_k5 <- s2_v2_k5$fit

chol_ok <- tryCatch({ chol(stage2_k5$Psi); TRUE }, error = function(e) FALSE)

cat("\nmethod:", s2_v2_k5$method, " converged:", isTRUE(stage2_k5$converged), "\n")
cat("cholesky:", chol_ok, " df.residual:", stage2_k5$df.residual, "\n")
cat("n coef:", length(coef(stage2_k5)), " (expect 20 = 4 terms x 5 outcomes)\n")
cat("\ncoef:\n")
print(round(coef(stage2_k5), 3))
cat("\npred_df:\n")
print(s2_v2_k5$pred_df[, .(CMA, PC1, PC2, PC3)])

  formula: cbind(theta1, theta2, theta3, theta4, theta5) ~ PC1 + PC2 + PC3 | random: TRUE | method: reml

method: reml  converged: TRUE 
cholesky: TRUE  df.residual: -10 
n coef: 20  (expect 20 = 4 terms x 5 outcomes)

coef:
theta1.(Intercept) theta2.(Intercept) theta3.(Intercept) theta4.(Intercept) 
            -0.872             -4.606             -4.481             -3.813 
theta5.(Intercept)         theta1.PC1         theta2.PC1         theta3.PC1 
            -1.020             -1.319             -0.538             -0.872 
        theta4.PC1         theta5.PC1         theta1.PC2         theta2.PC2 
             0.165             -1.448              0.146              0.433 
        theta3.PC2         theta4.PC2         theta5.PC2         theta1.PC3 
            -0.076              0.525             -0.748              0.737 
        theta2.PC3         theta3.PC3         theta4.PC3         theta5.PC3 
             0.838              0.832              0.462              1.118 

pred

formula: cbind(theta1, theta2, theta3, theta4, theta5) ~ PC1 + PC2 + PC3 | random: TRUE | method: reml

method: reml  converged: TRUE
cholesky: TRUE  df.residual: -10
n coef: 20  (expect 20 = 4 terms x 5 outcomes)

coef:
theta1.(Intercept) theta2.(Intercept) theta3.(Intercept) theta4.(Intercept)
            -0.872             -4.606             -4.481             -3.813
theta5.(Intercept)         theta1.PC1         theta2.PC1         theta3.PC1
            -1.020             -1.319             -0.538             -0.872
        theta4.PC1         theta5.PC1         theta1.PC2         theta2.PC2
             0.165             -1.448              0.146              0.433
        theta3.PC2         theta4.PC2         theta5.PC2         theta1.PC3
            -0.076              0.525             -0.748              0.737
        theta2.PC3         theta3.PC3         theta4.PC3         theta5.PC3
             0.838              0.832              0.462              1.118

pred_df:
         CMA       PC1         PC2        PC3
      <char>     <num>       <num>      <num>
1:   Toronto -1.998866 -0.71173777 -2.9258033
2:  Montreal -3.730649  0.95436513  1.8301977
3: Vancouver  1.354757 -3.38203234  1.6156235
4:    Ottawa  2.233846 -0.01748303 -1.0322194
5:    Quebec  2.140912  3.15688801  0.5122016

---

K=5 with PCs. formula ~ PC1 + PC2 + PC3, 20 coefficients, reml converged, cholesky TRUE

the K=3 cholesky failure  from previous now cleared. at K=3 all between-city spread lived in Ψ and pushed the
estimate to the boundary; PCs absorb part of it into fixed effects so Ψ lands inside the space

PC slopes non-zero — PC1 spans -1.45 to +0.17 across outcomes, PC3 +0.46 to +1.12. per-DA
theta can vary now, which is what the flat map needed

we predicted df.residual positive, got -10 where K=3 was -5. adding 3 predictors made it
worse not better — 25 observations against 20 fixed coefficients plus Ψ's free parameters, so
Q and I² still not computable

tarball

saving: 5 red, 5 res, diag5, stage2_k5, pred_df, cma_predictors, Z_means, 5 Z matrices,
and the ottawa/québec age tables — those last two cost 1.3gb of downloads and a grep over
14gb, everything else regenerates from seed in minutes

In [ ]:
diag5 <- rbindlist(list(tor_out$diag, mtl_out$diag, van_out$diag, ott_out$diag, qc_out$diag))

red5 <- list(Toronto = tor_out$red, Montreal = mtl_out$red, Vancouver = van_out$red,
             Ottawa = ott_out$red, Quebec = qc_out$red)
res5 <- list(Toronto = tor_out$res, Montreal = mtl_out$res, Vancouver = van_out$res,
             Ottawa = ott_out$res, Quebec = qc_out$res)
ids5 <- list(Toronto = tor_out$da_ids, Montreal = mtl_out$da_ids, Vancouver = van_out$da_ids,
             Ottawa = ott_out$da_ids, Quebec = qc_out$da_ids)

save_to_drive(list(
  red5_v2        = red5,
  res5_v2        = res5,
  Z5_v2          = Z_list,
  ids5_v2        = ids5,
  diag5_v2       = diag5,
  stage2_k5_v2   = list(fit = stage2_k5, pred_df = s2_v2_k5$pred_df,
                        method = s2_v2_k5$method, formula = s2_v2_k5$formula),
  cma_predictors = cma_predictors,
  new_age_ottqc  = new_age
))

chk_red <- readRDS("/content/saves_eod/red5_v2.rds")
chk_age <- readRDS("/content/saves_eod/new_age_ottqc.rds")
cat("\nred5 cities:", length(chk_red), " names:", paste(names(chk_red), collapse=" "), "\n")
cat("theta lengths:", paste(sapply(chk_red, function(x) length(x$theta_star)), collapse=" "), "\n")
cat("age tables:", paste(names(chk_age), collapse=" "),
    " rows:", paste(sapply(chk_age, nrow), collapse=" "), "\n")
print(diag5[, .(cma, n_da, deaths, sliver_deaths, ref_temp, qaic_A)])

saved 8 objects -> /content/drive/MyDrive/thesis/dlnm-pilot/saves_eod_2026-07-22.tar.gz 

red5 cities: 5  names: Toronto Montreal Vancouver Ottawa Quebec 
theta lengths: 5 5 5 5 5 
age tables: Ottawa Quebec  rows: 2045 1317 
         cma  n_da deaths sliver_deaths ref_temp   qaic_A
      <char> <int>  <int>         <int>    <num>    <num>
1:   Toronto  7682 610216          2419 19.38333 27047.04
2:  Montreal  6504 203988          1524 19.00753 27967.07
3: Vancouver  3573  56157           649 17.04646 27441.62
4:    Ottawa  2042  51476          1247 18.58291 28992.71
5:    Quebec  1310  26977           982 16.79456 28083.29


saved 8 objects -> /content/drive/MyDrive/thesis/dlnm-pilot/saves_eod_2026-07-22.tar.gz

red5 cities: 5  names: Toronto Montreal Vancouver Ottawa Quebec
theta lengths: 5 5 5 5 5
age tables: Ottawa Quebec  rows: 2045 1317
         cma  n_da deaths sliver_deaths ref_temp   qaic_A
      <char> <int>  <int>         <int>    <num>    <num>
1:   Toronto  7682 610216          2419 19.38333 27047.04
2:  Montreal  6504 203988          1524 19.00753 27967.07
3: Vancouver  3573  56157           649 17.04646 27441.62
4:    Ottawa  2042  51476          1247 18.58291 28992.71
5:    Quebec  1310  26977           982 16.79456 28083.29

---

8 objects banked to saves_eod_2026-07-22, overwriting the morning K=3 tarball. read-back
5 cities all 5-vectors, age tables 2045 and 1317 matching their DA lists

five-city diag: deaths 610216 → 26977, ref_temp 19.4 → 16.8, qaic_A 27047-28993 tight band

ottawa 2042 DAs gives 1247 sliver deaths where vancouver's 3573 DAs gives 649—sliver is
always 150 DAs, so this is per-DA 75-84 population not city size